<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="Rockborne logo" />
</center>

# Automated Exploratory Data Analysis with Sweetviz

*Student version*

## Overview

In the previous EDA lecture we worked through the manual logic of exploratory analysis: inspect the data, check quality, summarise variables, compare groups, build charts, and turn the evidence into conclusions.

This notebook keeps that same workflow, but adds **Sweetviz** as an automated EDA tool.

Sweetviz is useful because it gives us a quick first profile of a dataset. It can show column types, missing values, distributions, associations and useful comparisons. It does not replace the analyst. It gives the analyst a faster starting point.

## Learning objectives

By the end of this notebook you will be able to:

- Explain where automated EDA fits in the normal analysis workflow.
- Install and import Sweetviz in Databricks.
- Load data from `config.py` using Spark, then convert it to pandas for Sweetviz.
- Build a simple Sweetviz report with `sv.analyze()`.
- Build a target-focused Sweetviz report when there is a clear outcome field.
- Read a Sweetviz report in a structured way.
- Use pandas tables and charts to follow up on what the report highlights.
- Complete a live Superstore EDA lab in progressive exercise chunks.

## Prerequisites

You should already be comfortable with:

- Basic pandas inspection methods such as `.head()`, `.shape`, `.info()`, `.describe()` and `.value_counts()`.
- Selecting and filtering columns.
- Creating new columns.
- Grouping and aggregating with `.groupby()`.
- Creating simple Matplotlib or Seaborn charts.
- Explaining what a table or chart means in plain English.

## Index

- [Section 0: Setup](#setup)
- [Section 1: Why automated EDA?](#why)
- [Section 2: Load data with Spark from config.py](#load)
- [Section 3: First inspection](#first)
- [Section 4: Titanic walkthrough](#titanic)
- [Section 5: Reading a Sweetviz report](#reading)
- [Section 6: Superstore live class lab](#superstore-lab)
- [Section 7: Next steps after automated EDA](#next-steps)
- [Section 8: Wrap up and further exploration](#wrapup)

<a id="setup"></a>
# Section 0: Setup

Sweetviz is the only extra library installed in this notebook.

Run the install cell before importing Sweetviz. If Sweetviz has already been imported in the current session with another version, restart the Python session once, then run the notebook from the top.

In [ ]:
%pip install sweetviz==2.3.1

## Import libraries

We keep the imports in one place so we can see the tools used in the notebook.

In [ ]:
# pandas is used for data inspection and follow up analysis.
# Matplotlib and Seaborn are used for supporting charts.
# Sweetviz is used for automated EDA reports.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv

In [ ]:
# The file in Databricks should be named config.py.
# It sits in the same folder as this notebook.
# session_datasets contains the S3 paths used in the session.

from config import session_datasets

In [ ]:
# Check the dataset names available from config.py.
# This is a quick confidence check before reading any data.

session_datasets.keys()

<a id="why"></a>
# Section 1: Why automated EDA?

Exploratory Data Analysis is an investigation. It helps us understand what is in the data before we build a model, dashboard or recommendation.

A normal EDA workflow asks:

- What is the shape of the data?
- What does each column mean?
- Which fields are numeric, categorical or date-like?
- Where are values missing?
- Are any values implausible?
- Which groups behave differently?
- Which findings need manual follow up?

Sweetviz helps with the first pass. It does not decide which findings matter. That is still the analyst's job.

## The pattern we will use

The notebook uses the same pattern repeatedly:

1. Load the data.
2. Inspect it manually.
3. Prepare only what is needed.
4. Run Sweetviz.
5. Read the report.
6. Follow up with pandas.
7. Explain the finding in plain English.

This is important for beginners because it prevents Sweetviz from becoming a button we press without understanding the output.

<a id="load"></a>
# Section 2: Load data with Spark from config.py

The data is stored in S3 and read inside Databricks.

We use Spark to read the CSV files, then convert the Spark DataFrames to pandas DataFrames. Sweetviz works with pandas, so the `.toPandas()` step is required.

In [ ]:
# We store the Titanic S3 path in a clear variable.
# This keeps the next read cell short and easy to understand

titanic_path = session_datasets["train_titanic"]

In [ ]:
# Print the Titanic path in this cell.

print(titanic_path)

In [ ]:
# Read Titanic with Spark, then convert to pandas.
# header=True uses the first row as column names.
# inferSchema=True lets Spark infer basic data types.

titanic_raw = spark.read.csv(
    titanic_path,
    header=True,
    inferSchema=True
).toPandas()

In [ ]:
# Confirm that Titanic loaded.
# Shape returns rows first, columns second.

print("Titanic data loaded")
print(titanic_raw.shape)

In [ ]:
# Store the Superstore S3 path in its own variable.
# We use this later for the class exercises.

superstore_path = session_datasets["superstore"]

In [ ]:
# Print the Superstore path in this cell.
# This keeps the data loading checks separated.

print(superstore_path)

In [ ]:
# Read Superstore with Spark, then convert to pandas.
# We use the same Spark bridge pattern as Titanic.

store_raw = spark.read.csv(
    superstore_path,
    header=True,
    inferSchema=True
).toPandas()

In [ ]:
# Confirm that Superstore loaded.
# We do this separately from Titanic so each output is easy to read.

print("Superstore data loaded")
print(store_raw.shape)

<a id="first"></a>
# Section 3: First inspection

Before using Sweetviz, we still inspect the raw data manually.

This confirms that the data loaded correctly and gives us a first look at the columns.

In [ ]:
# Look at the first few Titanic rows.
# This is a quick check of column names and example values.



In [ ]:
# Look at the first few Superstore rows.



In [ ]:
# List the Titanic columns.
# Column names matter because we will reference them directly in pandas.



In [ ]:
# List the Superstore columns.
# This helps learners recognise the business fields before the lab.



In [ ]:
# Check Titanic data types.
# This tells us whether Spark inferred numbers and text correctly.



In [ ]:
# Check Superstore data types.
# Dates often need extra attention after CSV loading.



### Quick check: two minutes

Use the raw DataFrames to answer these questions:

1. How many rows are in Titanic?
2. How many rows are in Superstore?
3. Which Titanic columns have missing values?
4. Which Superstore columns look like date fields?

In [ ]:
# Your turn.
# Hint 1: use .shape for rows and columns.
# Hint 2: use .isna().sum() for missing values.
# Hint 3: use .columns to look through field names.

<a id="titanic"></a>
# Section 4: Titanic walkthrough

Titanic is used first because the dataset is small and has a clear outcome column.

The aim is to teach the Sweetviz workflow before moving to the larger Superstore lab.

## Step 1: copy the raw data

We copy the raw DataFrame before changing anything.

This is a good habit because it preserves the original data loaded from S3.

In [ ]:
# Work on a copy.
# The original titanic_raw DataFrame remains unchanged.



## Step 2: standardise column names

Standardised column names make code easier to write and read.

For this notebook we use lowercase names and replace spaces with underscores.

In [ ]:
# Clean column names in one step.
# This avoids repeated typing of mixed-case column names.



In [ ]:
# Check that the column names changed as expected.
# This is a small but important validation step.



## Step 3: confirm the inferred data types

Spark used `inferSchema=True` when it read the CSV file. This means the main numeric columns should already be numeric.

We check the data types before changing anything. We only convert a column when the inspection shows that its current type is wrong.

In [ ]:
# Check the data types created by Spark.
# Survived, Pclass, Age and Fare should already be numeric.



## Step 4: build a simple Sweetviz report

We begin with the basic Sweetviz pattern.

There is no target column and no feature engineering at this stage. The purpose is to see what Sweetviz can learn from the dataset as it currently stands.

In [ ]:
# Create a general automated EDA report.
# Sweetviz profiles every column in the Titanic DataFrame.



In [ ]:
# Display the report inside the Databricks notebook.
# Open each feature in the report to inspect its distribution and missing values.



### Read the first report

Start with the report overview, then work through the features.

Look for:

1. The number of rows and columns.
2. Columns with missing values.
3. Numeric distributions that appear skewed.
4. Categorical fields with uneven group sizes.
5. Identifier fields that are useful for tracing records but less useful for analysis.

### Quick check: two minutes

Use pandas to confirm the missing values highlighted by Sweetviz.

Return only columns that contain at least one missing value, ordered from most missing to least missing.

In [ ]:
# Your turn.
# Hint 1: start with titanic.isna().sum().
# Hint 2: keep counts greater than zero.
# Hint 3: sort the result from highest to lowest.


## Step 5: build a target-focused Sweetviz report

A general report describes the whole dataset. A target-focused report adds a specific outcome that we want to understand.

For Titanic, `survived` is already a numeric 0 and 1 field, so no conversion is required. We first confirm its values, then use it as the target.

In [ ]:
# Check the target before using it.
# We expect only 0 and 1, with no unexpected values.



In [ ]:
# Build a report that focuses on survival.
# Sweetviz will show how the other features relate to the target.



In [ ]:
# Display the target-focused report in Databricks.
# Compare this report with the earlier general profile.



<a id="reading"></a>
# Section 5: Reading a Sweetviz report

Do not scroll through the report without a plan.

Use this reading order:

1. Start with the overview.
2. Check row count and feature count.
3. Check missing values.
4. Review the target balance when a target is used.
5. Look at the strongest relationships.
6. Select one or two findings for manual follow up.

The report helps us form questions. Pandas gives us the exact evidence needed to answer them.

## Follow up 1: survival by sex

The target report should draw attention to differences between passenger groups.

We now calculate the exact survival rate by sex so the finding can be explained with a clear number.

In [ ]:
# Count passengers and calculate the mean survival value for each group.
# The mean of a 0 and 1 column is the proportion that survived.



In [ ]:
# Add a percentage column for easier interpretation.
# The original decimal rate is kept for analysis.



In [ ]:
# A bar chart is suitable because we are comparing two categories.
# The table above provides the exact figures behind the chart.



## Follow up 2: survival by passenger class

Passenger class is another relationship worth validating.

We compare the number of passengers, survival rate and average fare for each class.

In [ ]:
# Summarise survival and fare by passenger class.
# Sorting by Pclass keeps the natural class order in the output.



In [ ]:
# Add a percentage version of the survival rate.
# Review passenger count before comparing percentages.



In [ ]:
# A bar chart makes the three passenger classes easy to compare.
# The y axis uses a percentage because that is easier to discuss.



### Quick check: two minutes

Sweetviz shows age as a continuous variable. For a short group comparison, create an `age_band` column and calculate survival by age band.

Use these groups:

- Child: 0 to 12
- Teen: over 12 to 18
- Young adult: over 18 to 35
- Adult: over 35 to 60
- Older adult: over 60 to 100

In [ ]:
# Your turn.
# Hint 1: use pd.cut() to create age_band.
# Hint 2: group by age_band.
# Hint 3: count survived and calculate its mean.


## Titanic section conclusion

Sweetviz gave us a fast view of the dataset and helped identify relationships worth investigating.

The pandas follow ups then gave us exact counts and rates. This is the main workflow for the rest of the notebook:

1. Profile broadly.
2. Select a useful finding.
3. Validate it manually.
4. Explain it in plain English.

<a id="superstore-lab"></a>
# Section 6: Superstore live class lab

Superstore is the main practical section.

The class will work in 15 minute chunks. At each exercise point, complete the task in the `Your turn` cell. The coach will then review the approach and take questions.

The investigation focuses on profitability and discount behaviour. The aim is to use Sweetviz for the first profile, then use pandas and charts for focused follow up.

## Lab chunk 1: prepare and inspect Superstore

### Task

Prepare `store_raw` for analysis.

You should:

1. Copy the raw DataFrame.
2. Standardise the column names.
3. Check the cleaned names.
4. Check the data types inferred by Spark.

Do not convert numeric columns automatically. Convert a column only when the data type check shows that it is needed.

In [ ]:
# Your turn.
# Hint 1: start with store = store_raw.copy().
# Hint 2: use string methods on store.columns.
# Hint 3: replace spaces and hyphens with underscores.
# Hint 4: use .dtypes to check the inferred types.


## Lab chunk 2: build the first Superstore report

### Task

Create and display a standard Sweetviz report for the prepared `store` DataFrame.

Do not set a target yet. This first report is a broad profile of the dataset.

In [ ]:
# Your turn.
# Hint 1: pass store into sv.analyze().
# Hint 2: save the result to a clearly named variable.
# Hint 3: call .show_notebook() in the next cell.


### Read the report

Use the report to identify questions rather than conclusions.

Focus on:

1. The distributions of sales, profit and discount.
2. The balance across segment, region and category.
3. The number of unique values in identifier fields.
4. Missing values or unusual values.
5. One relationship that should be checked with pandas.

## Lab chunk 3: create profitability flags

### Task

Create two simple indicators:

- `is_profitable`: 1 when profit is greater than zero, otherwise 0
- `is_loss`: 1 when profit is below zero, otherwise 0

These are teaching features. They make rates easy to calculate with `.mean()`.

In [ ]:
# Your turn.
# Hint 1: compare the profit column with zero.
# Hint 2: use .astype(int) to turn True and False into 1 and 0.
# Hint 3: check the result with value_counts().


## Lab chunk 4: investigate profitability by category

### Task

Build a category summary with:

- order lines
- total sales
- total profit
- average discount
- loss rate
- profit margin

Sort the result by loss rate from highest to lowest.

In [ ]:
# Your turn.
# Hint 1: group by category.
# Hint 2: count order_id for order lines.
# Hint 3: average is_loss to calculate the loss rate.
# Hint 4: calculate profit margin after the groupby.


## Lab chunk 5: build a target-focused Superstore report

### Task

Use `is_profitable` as the target in a second Sweetviz report.

This changes the question from "What is in the dataset?" to "Which features appear related to profitability?"

In [ ]:
# Your turn.
# Hint 1: use store as the source DataFrame.
# Hint 2: set target_feat="is_profitable".
# Hint 3: display the report in a separate cell.


### Read the target report

Focus on:

1. Whether discount appears related to profitability.
2. Whether category or sub-category stands out.
3. Whether any result is driven by an uneven group size.
4. Which finding should be tested with a grouped pandas table.

Sweetviz shows associations. It does not prove that one feature caused another.

## Lab chunk 6: investigate discount groups

### Task

Create five readable discount groups, then compare their loss rates and profit margins.

Use these groups:

- Full Retail: 0 discount
- Welcome Campaign: over 0 up to 20 percent
- Seasonal Clearance: over 20 up to 45 percent
- Mid Incentive Campaign: over 45 up to 60 percent
- Late Season Campaign: over 60 percent

In [ ]:
# Your turn.
# Hint 1: use pd.cut() on the discount column.
# Hint 2: provide five bin labels.
# Hint 3: group by the new discount_campaign field.
# Hint 4: calculate loss rate and profit margin.


## Lab chunk 7: write the business conclusion

### Task

Write a short conclusion using the category and discount summaries.

Include:

1. One clear finding.
2. One supporting number.
3. One caveat.
4. One recommended follow up.

In [ ]:
# Your turn.
# Hint 1: use category_summary.iloc[0] for the highest loss category.
# Hint 2: use discount_summary.iloc[0] for the highest loss discount group.
# Hint 3: keep the conclusion evidence-based and concise.


<a id="next-steps"></a>
# Section 7: Next steps after automated EDA

After a Sweetviz report, the next step is not to stop at the visual profile.

Use this follow up pattern:

1. Select one useful finding.
2. Build a pandas table to quantify it.
3. Add one chart when it improves communication.
4. Write a plain English conclusion.
5. State the main caveat.

For risk and analytics work, this is what turns an automated report into evidence that a stakeholder can use.

<a id="wrapup"></a>
# Section 8: Wrap up

In this notebook we used Sweetviz as one part of a complete EDA workflow.

The sequence was:

1. Read S3 data through Spark using paths from `config.py`.
2. Convert the Spark DataFrames to pandas.
3. Inspect the data types before changing anything.
4. Build a simple Sweetviz report.
5. Add a target only when the analysis had a clear outcome.
6. Validate report findings with pandas tables and charts.
7. Write an evidence-based conclusion.

The key lesson is that Sweetviz gives us a fast starting point. The analyst is still responsible for checking the evidence and explaining what it means.

## Final reflection questions

Use these questions to close the session:

1. What did Sweetviz show faster than manual EDA?
2. Why did we inspect data types before converting any columns?
3. How did the target-focused report differ from the general report?
4. Which Sweetviz finding needed manual validation?
5. What would you check before sharing an automated EDA report with a stakeholder?